<a href="https://colab.research.google.com/github/nashranoor98/credit-card-fraud-detection/blob/main/CaseStudy2.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

1. Loading and Studying Dataset

In [ ]:
import pandas as pd
import numpy as np

# Load the IEEE-CIS Fraud Detection dataset
tx = pd.read_csv("data/train_transaction.csv")
identity = pd.read_csv("data/train_identity.csv")

# Merge transaction and identity information
df = tx.merge(identity, on="TransactionID", how="left")

print("Dataset shape:", df.shape)
print(df["isFraud"].value_counts())

In [ ]:
print(df.head())
print("Missing values:", df.isnull().sum().sum())

2. Splitting Dataset

In [ ]:
from sklearn.model_selection import train_test_split

# Use a practical balanced working sample because the full IEEE-CIS
# dataset is large for SMOTE on memory-limited machines.
fraud = df[df["isFraud"] == 1]
normal = df[df["isFraud"] == 0].sample(n=50000, random_state=42)
df = pd.concat([normal, fraud]).sample(frac=1, random_state=42).reset_index(drop=True)

transaction_features = ["TransactionDT", "TransactionAmt", "card1", "card2", "card3", "card5", "addr1", "addr2", "dist1", "dist2"] + [f"C{i}" for i in range(1, 15)] + [f"D{i}" for i in range(1, 16)] + [f"V{i}" for i in range(1, 21)]
identity_features = identity.select_dtypes(include=np.number).columns.tolist()
identity_features = [c for c in identity_features if c != "TransactionID"]
feature_columns = [c for c in transaction_features + identity_features if c in df.columns]

X = df[feature_columns].replace([np.inf, -np.inf], np.nan)
y = df["isFraud"].astype(int)

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

print("Normal Cases:", sum(y_train == 0))
print("Fraud Cases:", sum(y_train == 1))

In [ ]:
from sklearn.impute import SimpleImputer

imputer = SimpleImputer(strategy="median")
X_train = imputer.fit_transform(X_train)
X_test = imputer.transform(X_test)

3. Applying SMOTE

In [ ]:
from imblearn.over_sampling import SMOTE

smote = SMOTE(random_state=42)
X_train_smote, y_train_smote = smote.fit_resample(X_train, y_train)

print("Normal cases:", sum(y_train_smote == 0))
print("Fraud cases:", sum(y_train_smote == 1))

4. Training XGBoost and checking Feature Importance

In [ ]:
import xgboost as xgb
import matplotlib.pyplot as plt

model = xgb.XGBClassifier(n_estimators=150, max_depth=6, learning_rate=0.08, subsample=0.8, colsample_bytree=0.8, eval_metric="logloss", random_state=42, n_jobs=2)
model.fit(X_train_smote, y_train_smote)

In [ ]:
plt.figure(figsize=(10, 8))
xgb.plot_importance(model, max_num_features=10)
plt.title("XGBoost Feature Importance")
plt.show()

5. Prediction

In [ ]:
from sklearn.metrics import roc_auc_score, precision_recall_fscore_support

probabilities = model.predict_proba(X_test)[:, 1]
print("ROC-AUC Score:", round(roc_auc_score(y_test, probabilities), 4))

# Tune the decision threshold using F1-score
thresholds = np.arange(0.10, 0.91, 0.05)
scores = []
for threshold in thresholds:
    pred = (probabilities >= threshold).astype(int)
    p, r, f1, _ = precision_recall_fscore_support(y_test, pred, average="binary", zero_division=0)
    scores.append((threshold, p, r, f1))

scores = pd.DataFrame(scores, columns=["threshold", "precision", "recall", "f1"])
best = scores.loc[scores["f1"].idxmax()]
custom_threshold = float(best["threshold"])
y_pred = (probabilities >= custom_threshold).astype(int)
print("Selected threshold:", custom_threshold)
print(scores)

6. Evaluating Model

In [ ]:
from sklearn.metrics import confusion_matrix, classification_report

print("Confusion Matrix:\n", confusion_matrix(y_test, y_pred))
print("\nClassification Report\n", classification_report(y_test, y_pred, digits=4))

The model uses SMOTE to handle the highly imbalanced fraud class. XGBoost learns non-linear patterns, while threshold tuning helps balance precision and recall. Feature importance shows which variables contributed most to the fitted model.